In [1]:
%run vendite.ipynb
import os

(909, 9)
       articolo     brand        fornitore modalita_acquisto categoria    colore stagione  prezzo_listino  costo_unitario
0  AGO-CAM-0001  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     verde     PE23           109.0           40.47
1  AGO-CAM-0002  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           119.0           47.71
2  AGO-CAM-0003  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  fantasia     PE23           129.0           48.24
3  AGO-CAM-0004  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23            79.0           30.54
4  AGO-CAM-0005  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia    bianco     PE23           139.0           52.70
5  AGO-CAM-0006  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           129.0           49.90
6  AGO-CRA-0007  Ago&Filo  Ago&Filo S.r.l.             fermo  cravatta     rosso     PE23            79.0           31.77
7  AGO-CRA-0008

In [2]:
rng_a = np.random.default_rng(2024)
 
V = vendite.copy()
O = ordini.copy()
O.insert(0, "id_ordine", range(1, len(O) + 1))
 
registro = []
 
def registra(file, ids, tipo, famiglia, colonna, prima, dopo):
    for i, p, d in zip(ids, prima, dopo):
        registro.append({"file": file, "id": int(i), "tipo": tipo, "famiglia": famiglia,
                         "colonna": colonna, "prima": p, "dopo": d})

def scegli(maschera, n):
    """n righe a caso tra quelle che soddisfano la maschera (indici del DataFrame)."""
    candidati = np.flatnonzero(np.asarray(maschera))
    return rng_a.choice(candidati, size=min(n, len(candidati)), replace=False)

In [3]:
vend = V["tipo_movimento"] == "vendita"
giorno = V["data_ora"].dt.normalize()

vietati = giorno.isin(cal.loc[cal["divieto_promo"], "data"])
idx = scegli(vend & vietati & (V["motivo_sconto"] == "nessuno") & (V["categoria"] != "profumo"), 15)
prima = V.loc[idx, "prezzo_venduto"].tolist()
V.loc[idx, ["sconto_pct", "motivo_sconto"]] = [20, "saldo"]
V.loc[idx, "prezzo_venduto"] = (V.loc[idx, "prezzo_listino"] * 0.8).round(2)
registra("vendite", V.loc[idx, "id_riga"], "promo_periodo_vietato", "segnalare", "prezzo_venduto",
         prima, V.loc[idx, "prezzo_venduto"])

idx = scegli(vend & (V["motivo_sconto"] == "cliente_abituale"), 12)
nuovo = rng_a.choice([20, 25, 30], size=len(idx))
prima = V.loc[idx, "sconto_pct"].tolist()
V.loc[idx, "sconto_pct"] = nuovo
V.loc[idx, "prezzo_venduto"] = (V.loc[idx, "prezzo_listino"] * (1 - nuovo / 100)).round(2)
registra("vendite", V.loc[idx, "id_riga"], "sconto_oltre_tetto", "segnalare", "sconto_pct", prima, nuovo)

saldo_del_giorno = V["data_ora"].dt.normalize().map(cal.set_index("data")["stagione_in_saldo"])
sconto_del_giorno = V["data_ora"].dt.normalize().map(cal.set_index("data")["sconto_saldo"])
idx = scegli(vend & (V["categoria"] == "profumo") & (V["motivo_sconto"] == "nessuno")
             & (saldo_del_giorno == V["stagione"]), 5)
prima = V.loc[idx, "prezzo_venduto"].tolist()
V.loc[idx, "sconto_pct"] = (sconto_del_giorno[idx] * 100).round().astype(int)
V.loc[idx, "motivo_sconto"] = "saldo"
V.loc[idx, "prezzo_venduto"] = (V.loc[idx, "prezzo_listino"] * (1 - sconto_del_giorno[idx])).round(2)
registra("vendite", V.loc[idx, "id_riga"], "profumo_in_saldo", "segnalare", "prezzo_venduto",
         prima, V.loc[idx, "prezzo_venduto"])

idx = scegli(vend & (V["data_ora"].dt.day_of_week == 0), 6)
prima = V.loc[idx, "data_ora"].tolist()
V.loc[idx, "data_ora"] = V.loc[idx, "data_ora"].dt.normalize() + pd.to_timedelta(
    rng_a.integers(10 * 60, 13 * 60, size=len(idx)), unit="min")
registra("vendite", V.loc[idx, "id_riga"], "vendita_negozio_chiuso", "segnalare", "data_ora",
         prima, V.loc[idx, "data_ora"])

In [4]:
os.makedirs("../valutazione", exist_ok=True)
V.to_csv("../valutazione/vendite_riferimento.csv", index=False)
O.to_csv("../valutazione/ordini_riferimento.csv", index=False)

In [5]:
D = V.copy().astype({"taglia": object, "brand": object, "colore": object,
                     "prezzo_venduto": object, "sconto_pct": object, "motivo_sconto": object})
n = len(D)

NOMI_TAGLIE = {"S": "Small", "M": "Medium", "L": "Large", "XL": "Extra Large"}
idx = scegli(D["taglia"].notna() & (D["taglia"] != "TU"), int(n * 0.03))
prima = D.loc[idx, "taglia"].tolist()
dopo = [rng_a.choice([t.lower(), t + " ", NOMI_TAGLIE.get(t, "tg " + t)]) for t in prima]
D.loc[idx, "taglia"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "taglia_incoerente", "pulire", "taglia", prima, dopo)

idx = scegli(D["brand"].notna(), int(n * 0.02))
prima = D.loc[idx, "brand"].tolist()
dopo = [rng_a.choice([b.upper(), b + " ", b.lower()]) for b in prima]
D.loc[idx, "brand"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "brand_incoerente", "pulire", "brand", prima, dopo)

idx = scegli(D["colore"].notna() & (D["colore"] != "n.d."), int(n * 0.01))
prima = D.loc[idx, "colore"].tolist()
dopo = [rng_a.choice([c.upper(), c.capitalize(), "blu navy" if c == "blu" else c + " "]) for c in prima]
D.loc[idx, "colore"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "colore_incoerente", "pulire", "colore", prima, dopo)

idx = scegli(D["prezzo_venduto"].notna(), int(n * 0.02))
prima = D.loc[idx, "prezzo_venduto"].tolist()
dopo = [f"{p:.2f}".replace(".", ",") for p in prima]
D.loc[idx, "prezzo_venduto"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "prezzo_testo_virgola", "pulire", "prezzo_venduto", prima, dopo)

liberi = D["prezzo_venduto"].map(lambda x: isinstance(x, float))
idx = scegli(liberi, int(n * 0.005))
prima = D.loc[idx, "prezzo_venduto"].tolist()
D.loc[idx, "prezzo_venduto"] = None
registra("vendite", D.loc[idx, "id_riga"], "prezzo_mancante", "pulire", "prezzo_venduto", prima, [None] * len(idx))

liberi = D["prezzo_venduto"].map(lambda x: isinstance(x, float) and x == x)
idx = scegli(liberi, 8)
prima = D.loc[idx, "prezzo_venduto"].tolist()
dopo = [-p for p in prima]
D.loc[idx, "prezzo_venduto"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "prezzo_negativo", "pulire", "prezzo_venduto", prima, dopo)

idx = scegli(D["tipo_movimento"] == "vendita", 5)
prima = D.loc[idx, "sconto_pct"].tolist()
dopo = rng_a.choice([150, -10, 300], size=len(idx)).tolist()
D.loc[idx, "sconto_pct"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "sconto_fuori_range", "pulire", "sconto_pct", prima, dopo)

idx = scegli((D["tipo_movimento"] == "vendita") & (D["sconto_pct"].isin(range(0, 101))), int(n * 0.005))
prima = D.loc[idx, "sconto_pct"].tolist()
dopo = [s + 10 if s <= 60 else s - 10 for s in prima]
D.loc[idx, "sconto_pct"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "sconto_incoerente", "pulire", "sconto_pct", prima, dopo)

idx = scegli(D["motivo_sconto"].isin(["saldo", "cliente_abituale"]), int(n * 0.005))
prima = D.loc[idx, "motivo_sconto"].tolist()
D.loc[idx, "motivo_sconto"] = None
registra("vendite", D.loc[idx, "id_riga"], "motivo_mancante", "pulire", "motivo_sconto", prima, [None] * len(idx))

idx = scegli(D["tipo_movimento"] == "vendita", 8)
prima = D.loc[idx, "articolo"].tolist()
dopo = [a[:-4] + "9" + f"{rng_a.integers(1000):03d}" for a in prima]
D.loc[idx, "articolo"] = dopo
registra("vendite", D.loc[idx, "id_riga"], "articolo_orfano", "pulire", "articolo", prima, dopo)

testo = D["data_ora"].dt.strftime("%Y-%m-%d %H:%M:%S")
idx = scegli(np.ones(n, bool), 3)
prima = testo[idx].tolist()
dopo = [rng_a.choice(["2031", "2019"]) + t[4:] for t in prima]
testo[idx] = dopo
registra("vendite", D.loc[idx, "id_riga"], "data_fuori_periodo", "pulire", "data_ora", prima, dopo)
idx = scegli(~testo.index.isin(idx), int(n * 0.03))
prima = testo[idx].tolist()
dopo = pd.to_datetime(testo[idx]).dt.strftime("%d/%m/%Y %H:%M").tolist()
testo[idx] = dopo
registra("vendite", D.loc[idx, "id_riga"], "data_formato_misto", "pulire", "data_ora", prima, dopo)
D["data_ora"] = testo

idx = scegli(np.ones(n, bool), int(n * 0.01))
registra("vendite", D.loc[idx, "id_riga"], "riga_duplicata", "pulire", "tutte", [None] * len(idx), [None] * len(idx))
D = pd.concat([D, D.loc[idx]]).sort_values("id_riga", kind="stable", ignore_index=True)

In [6]:
E = O.copy()
idx = scegli(np.ones(len(E), bool), 5)
prima = E.loc[idx, "qta_ricevuta"].tolist()
E.loc[idx, "qta_ricevuta"] = E.loc[idx, "qta_ordinata"] + rng_a.integers(1, 6, size=len(idx))
registra("ordini", E.loc[idx, "id_ordine"], "ricevuta_oltre_ordinata", "pulire", "qta_ricevuta",
         prima, E.loc[idx, "qta_ricevuta"])
 
idx = scegli(np.ones(len(E), bool), 3)
prima = E.loc[idx, "costo_unitario"].tolist()
E.loc[idx, "costo_unitario"] = (E.loc[idx, "prezzo_listino"] * 1.2).round(2)
registra("ordini", E.loc[idx, "id_ordine"], "costo_oltre_listino", "pulire", "costo_unitario",
         prima, E.loc[idx, "costo_unitario"])
 
idx = scegli(np.ones(len(E), bool), 3)
prima = E.loc[idx, "data_arrivo"].tolist()
E.loc[idx, ["data_ordine", "data_arrivo"]] = E.loc[idx, ["data_arrivo", "data_ordine"]].to_numpy()
registra("ordini", E.loc[idx, "id_ordine"], "arrivo_prima_ordine", "pulire", "data_arrivo",
         prima, E.loc[idx, "data_arrivo"])
 
idx = scegli(np.ones(len(E), bool), 5)
prima = E.loc[idx, "prezzo_listino"].tolist()
E.loc[idx, "prezzo_listino"] = E.loc[idx, "prezzo_listino"] + 10
registra("ordini", E.loc[idx, "id_ordine"], "listino_diverso_tra_file", "pulire", "prezzo_listino",
         prima, E.loc[idx, "prezzo_listino"])
 
idx = scegli(np.ones(len(E), bool), 3)
registra("ordini", E.loc[idx, "id_ordine"], "riga_duplicata", "pulire", "tutte", [None] * 3, [None] * 3)
E = pd.concat([E, E.loc[idx]]).sort_values("id_ordine", kind="stable", ignore_index=True)

In [7]:
os.makedirs("../data", exist_ok=True)
D.to_csv("../data/vendite.csv", index=False)
E.to_csv("../data/ordini.csv", index=False)
reg = pd.DataFrame(registro)
reg.to_csv("../valutazione/registro_anomalie.csv", index=False)
 
print("vendite.csv:", D.shape, "— ordini.csv:", E.shape)
print("\nAnomalie piantate:")
print(reg.groupby(["famiglia", "file", "tipo"]).size().to_string())

prova = pd.read_csv("../data/vendite.csv")
print("\nTipi di dato letti dal CSV (prezzo e data devono risultare 'sporchi', cioè testo):")
print(prova.dtypes[["data_ora", "prezzo_venduto", "sconto_pct", "taglia"]])
print("\nRighe duplicate trovate:", prova.duplicated().sum())

vendite.csv: (16569, 20) — ordini.csv: (4514, 16)

Anomalie piantate:
famiglia   file     tipo                    
pulire     ordini   arrivo_prima_ordine           3
                    costo_oltre_listino           3
                    listino_diverso_tra_file      5
                    ricevuta_oltre_ordinata       5
                    riga_duplicata                3
           vendite  articolo_orfano               8
                    brand_incoerente            328
                    colore_incoerente           164
                    data_formato_misto          492
                    data_fuori_periodo            3
                    motivo_mancante              82
                    prezzo_mancante              82
                    prezzo_negativo               8
                    prezzo_testo_virgola        328
                    riga_duplicata              164
                    sconto_fuori_range            5
                    sconto_incoerente            82
 